# DistilBERT-base-uncased (Pretrained Model) 

This notebook fine-tunes `distilbert-base-uncased`, a smaller, faster, and lighter version of the BERT-base model that retains 97% of BERT's language understanding while reducing size by 40% and increasing inference speed by 60%. It is "uncased", meaning it converts all text to lowercase and does not tell the difference between capital and small letters. 

# Importing Libraries

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
import wandb
from kaggle_secrets import UserSecretsClient
from dataclasses import dataclass, asdict
from gensim.models import Word2Vec

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

sns.set_theme(style="whitegrid", palette="muted")

Using device: cuda


# Loading Dataset

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print("Datasets loaded!")

Datasets loaded!


# Define the Model

## Configuration

In [8]:
@dataclass
class DBConfig:
    model_name: str = "distilbert-base-uncased"
    max_seq_len: int = 192
    epochs: int = 5
    batch_size: int = 8
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    wandb_project: str = "23f2004791-t22026"
    wandb_run_name: str = "distilbert-finetuned-1"

    def to_dict(self):
        return asdict(self)

db_cfg = DBConfig()

In [9]:
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
    print("Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"W&B Login Failed.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adrija935 (23f2004791-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into Weights & Biases!


## Tokenizer and Vocabulary Builder

In [10]:
db_tokenizer = AutoTokenizer.from_pretrained(db_cfg.model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## PyTorch Dataset and DataLoaders

In [11]:
class DBMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=192, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        option_texts = [str(row[opt]) for opt in self.options]

        encoded = self.tokenizer(
            [prompt] * len(self.options),
            option_texts,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoded['input_ids'],
            'attention_mask': encoded['attention_mask'] 
        }

        if not self.is_test:
            item['labels'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
        return item

# Clean the prompts to extract the core topic
prefixes_to_remove = [
    r"Pick the best possible answer: ",
    r"Determine the correct option: ",
    r"Select the most accurate option: ",
    r"Identify the correct statement: ",
    r"Which of the following is correct\? ",
    r" among the listed options\."
]

def clean_prompt(text):
    for prefix in prefixes_to_remove:
        text = re.sub(prefix, '', text, flags=re.IGNORECASE)
    return text.strip()

train_df['topic_group'] = train_df['prompt'].apply(clean_prompt)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(train_df, groups=train_df['topic_group']))

db_train_split = train_df.iloc[train_idx].reset_index(drop=True)
db_val_split = train_df.iloc[val_idx].reset_index(drop=True)

db_train_dataset = DBMCQDataset(db_train_split, db_tokenizer, max_len=db_cfg.max_seq_len)
db_val_dataset = DBMCQDataset(db_val_split, db_tokenizer, max_len=db_cfg.max_seq_len)
db_test_dataset = DBMCQDataset(test_df, db_tokenizer, max_len=db_cfg.max_seq_len, is_test=True)

db_train_loader = DataLoader(db_train_dataset, batch_size=db_cfg.batch_size, shuffle=True)
db_val_loader = DataLoader(db_val_dataset, batch_size=db_cfg.batch_size, shuffle=False)
db_test_loader = DataLoader(db_test_dataset, batch_size=db_cfg.batch_size, shuffle=False)

print(f"Train topics: {db_train_split['topic_group'].nunique()}")
print(f"Val topics: {db_val_split['topic_group'].nunique()}")

Train topics: 752
Val topics: 188


## Model 

In [12]:
db_model = AutoModelForMultipleChoice.from_pretrained(db_cfg.model_name).to(db_cfg.device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Scoring Metric (MAP@3)

In [13]:
# MAP@3 Metric
def compute_map_at_3(predictions, targets):
    scores = []
    for top_preds, target in zip(predictions, targets):
        score = 0.0
        for rank, pred in enumerate(top_preds):
            if pred == target:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

# Training and Validation

In [14]:
wandb.init(
    project=db_cfg.wandb_project,
    name=db_cfg.wandb_run_name,
    config=db_cfg.to_dict(),
    reinit=True
)

optimizer = torch.optim.AdamW(db_model.parameters(), lr=db_cfg.learning_rate, weight_decay=db_cfg.weight_decay)

total_steps = len(db_train_loader) * db_cfg.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * db_cfg.warmup_ratio),
    num_training_steps=total_steps
)

best_map3 = 0.0

for epoch in range(db_cfg.epochs):
    db_model.train()
    running_loss = 0.0

    for batch in db_train_loader:
        input_ids = batch['input_ids'].to(db_cfg.device)
        attention_mask = batch['attention_mask'].to(db_cfg.device)
        labels = batch['labels'].to(db_cfg.device)

        optimizer.zero_grad()
        outputs = db_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(db_model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * input_ids.size(0)

    train_loss = running_loss / len(db_train_loader.dataset)

    # Validation loop
    db_model.eval()
    val_loss = 0.0
    all_top3_preds = []
    all_top1_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in db_val_loader:
            input_ids = batch['input_ids'].to(db_cfg.device)
            attention_mask = batch['attention_mask'].to(db_cfg.device)
            labels = batch['labels'].to(db_cfg.device)

            outputs = db_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item() * input_ids.size(0)
            
            top1_preds = torch.argmax(outputs.logits, dim=1)
            all_top1_preds.append(top1_preds.cpu().numpy())
            _, top3_indices = torch.topk(outputs.logits, k=3, dim=1)
            all_top3_preds.append(top3_indices.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    val_loss = val_loss / len(db_val_loader.dataset)
    all_top1_preds = np.concatenate(all_top1_preds, axis=0)
    all_top3_preds = np.concatenate(all_top3_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    val_map3 = compute_map_at_3(all_top3_preds, all_targets)
    val_accuracy = accuracy_score(all_targets, all_top1_preds)
    val_f1 = f1_score(all_targets, all_top1_preds, average='macro')
    
    print(f"Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")
    
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map3": val_map3,
        "val_accuracy": val_accuracy,
        "val_f1": val_f1
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        torch.save(db_model.state_dict(), "best_distilbert_model.pt")
        print(f"New best model saved with MAP@3: {best_map3:.4f}")
        wandb.run.summary["best_map3"] = best_map3
        wandb.run.summary["best_accuracy"] = val_accuracy
        wandb.run.summary["best_f1"] = val_f1  

wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260726_134536-mboep3ds
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert-finetuned-1
wandb: ⭐️ View project at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: 🚀 View run at https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/mboep3ds


Epoch 1 | Val Loss: 0.9552 | Val MAP@3: 0.8163 | Val Acc: 0.7187 | Val F1: 0.7216
New best model saved with MAP@3: 0.8163
Epoch 2 | Val Loss: 0.4306 | Val MAP@3: 0.9233 | Val Acc: 0.8824 | Val F1: 0.8853
New best model saved with MAP@3: 0.9233
Epoch 3 | Val Loss: 0.2797 | Val MAP@3: 0.9633 | Val Acc: 0.9514 | Val F1: 0.9537
New best model saved with MAP@3: 0.9633
Epoch 4 | Val Loss: 0.2127 | Val MAP@3: 0.9638 | Val Acc: 0.9540 | Val F1: 0.9569
New best model saved with MAP@3: 0.9638
Epoch 5 | Val Loss: 0.1814 | Val MAP@3: 0.9672 | Val Acc: 0.9591 | Val F1: 0.9627


wandb: updating run metadata


New best model saved with MAP@3: 0.9672


wandb: uploading history steps 4-4, summary, console lines 8-9
wandb: 
wandb: Run history:
wandb:        epoch ▁▃▅▆█
wandb:   train_loss █▄▂▁▁
wandb: val_accuracy ▁▆███
wandb:       val_f1 ▁▆███
wandb:     val_loss █▃▂▁▁
wandb:     val_map3 ▁▆███
wandb: 
wandb: Run summary:
wandb: best_accuracy 0.95908
wandb:       best_f1 0.96265
wandb:     best_map3 0.96718
wandb:         epoch 5
wandb:    train_loss 0.17494
wandb:  val_accuracy 0.95908
wandb:        val_f1 0.96265
wandb:      val_loss 0.18142
wandb:      val_map3 0.96718
wandb: 
wandb: 🚀 View run distilbert-finetuned-1 at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026/runs/mboep3ds
wandb: ⭐️ View project at: https://wandb.ai/23f2004791-dl-genai-project/23f2004791-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260726_134536-mboep3ds/logs


# Inference

In [15]:
db_model.load_state_dict(torch.load("best_distilbert_model.pt"))
db_model.eval()

idx_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
submission_preds = []

with torch.no_grad():
    for batch in db_test_loader:
        input_ids = batch['input_ids'].to(db_cfg.device)
        attention_mask = batch['attention_mask'].to(db_cfg.device)

        outputs = db_model(input_ids=input_ids, attention_mask=attention_mask)
        _, top3_indices = torch.topk(outputs.logits, k=3, dim=1)

        for top_three in top3_indices.cpu().numpy():
            pred_str = " ".join([idx_to_label[i] for i in top_three])
            submission_preds.append(pred_str)

# Generate Submission

In [16]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to 'submission.csv'!")
print(submission_df.head())

Submission saved to 'submission.csv'!
   id Prediction
0   1      A E B
1   2      B A D
2   3      B D E
3   4      E D C
4   5      C A D
